--------------------------
#### Semantic text search using embeddings
------------------------------
- Semantic Search Efficiency:
    - Utilizing embeddings allows for efficient semantic search through all reviews.
    - The process involves embedding the search query and then identifying the most similar reviews.
    - This method enables quick retrieval of relevant reviews based on semantic similarity.
    
- Low Cost:
    - The cost-effectiveness of the search process is emphasized.
    - Embedding-based search minimizes computational expenses while maintaining search accuracy.
    - This approach offers an economical solution for exploring reviews in a semantically meaningful way.

In [1]:
import pandas as pd
import numpy as np

from ast import literal_eval

#### load the saved embeddings (food reviews)

In [2]:
datafile_path = r"D:\Makesh\Working\AI\RPS\Day07\Dataset\FoodReview\amazon_food_reviews_with_embeddings_2k.csv"

In [3]:
df = pd.read_csv(datafile_path)
df.shape

(2000, 9)

In [4]:
df.sample(3)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding
474,272984,B007PA33NY,A10E6BYP1D0I99,5,Love this BAM of a coffee,I love this coffee. I usually drink Coffee Peo...,Title: Love this BAM of a coffee; Content: I l...,83,"[-0.007256133481860161, -0.013239020481705666,..."
1956,32931,B001P05K8Q,A3L0B5NBTQ7ZHO,4,Great results but they stink,We originally purchased these chews from our v...,Title: Great results but they stink; Content: ...,128,"[0.05671019107103348, -0.002936408156529069, -..."
120,86950,B0030JFYYA,A2IDRAYYZ5GKBZ,5,Superb addition to a dog's daily diet,This Preference veggie mix has been an importa...,Title: Superb addition to a dog's daily diet; ...,368,"[0.001065522781573236, -0.023760739713907242, ..."


In [5]:
df.dtypes

Unnamed: 0        int64
ProductId        object
UserId           object
Score             int64
Summary          object
Text             object
combined         object
n_tokens          int64
ada_embedding    object
dtype: object

In [6]:
%%time
# convert string to array
df["embedding"] = df.ada_embedding.apply(literal_eval).apply(np.array)

CPU times: total: 8.73 s
Wall time: 9.47 s


#### Search query

- compare the cosine similarity of the embeddings of the query and the documents, and show top_n best matches.

In [7]:
from openai import OpenAI
import json
import os

In [8]:
openai_api_key = os.environ.get('OPENAI_API_KEY')

In [9]:
client = OpenAI()

In [10]:
# models
EMBEDDING_MODEL = "text-embedding-3-small"
GPT_MODEL       = "gpt-3.5-turbo"

In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_similarity(document_embeddings, query_embedding):
    """
    Calculate cosine similarity between the query embedding and document embeddings.

    Parameters:
    - query_embedding: The embedding vector for the search query.
    - document_embeddings: List of embeddings for the documents.

    Returns:
    - List of cosine similarities between the query and each document.
    """
    similarities = cosine_similarity(query_embedding, document_embeddings)
    return similarities.flatten()

In [13]:
# For a specific query and documents
query_embedding     = np.array([[0.1, 0.3, 0.5]])  

document_embeddings = np.array([
    [0.2, 0.4, 0.6],  
    [0.15, 0.35, 0.55],
   
])

In [14]:
query_embedding.reshape(-1, 1)

array([[0.1],
       [0.3],
       [0.5]])

In [15]:
get_cosine_similarity(query_embedding, document_embeddings)

array([0.99385869, 0.99808276])

In [17]:
# search through the reviews for a specific product
def get_sim_scores(df, query):
    
    response = client.embeddings.create(
                    model          = EMBEDDING_MODEL, 
                    input          = query, 
                    encoding_format= "float"
    )
    
    # obtain the embedding for the query
    query_embedding = np.array(response.data[0].embedding)[np.newaxis, :]
       
    df["similarity"] = df.embedding.apply(lambda x: get_cosine_similarity(x[np.newaxis, :], query_embedding))
    
    return df

In [18]:
df_with_sim_scores = get_sim_scores(df, "delicious beans")

In [32]:
df_with_sim_scores

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding,embedding,similarity
0,197286,B000PG8KGU,A30NLBXH8SIFSF,5,Party Pleasure!,"Perfect for a cocktail party! You get your choice of cashews, peanuts and honey roasted. Just put these snack size packages in a party bowl or on a party platter. You pick the snack, that you want, without everyone putting their bare hands on unopened choice of snacks. Not everyone washes their ...","Title: Party Pleasure!; Content: Perfect for a cocktail party! You get your choice of cashews, peanuts and honey roasted. Just put these snack size packages in a party bowl or on a party platter. You pick the snack, that you want, without everyone putting their bare hands on unopened choice of s...",110,"[0.025441810488700867, -0.020616639405488968, -0.030998067930340767, 0.027035579085350037, 0.0005725321243517101, -0.045912232249975204, -0.004744751378893852, 0.06246403232216835, -0.009584544226527214, 0.024432910606265068, 0.05734642595052719, -0.041671931743621826, 0.026494575664401054, -0.0...","[0.025441810488700867, -0.020616639405488968, -0.030998067930340767, 0.027035579085350037, 0.0005725321243517101, -0.045912232249975204, -0.004744751378893852, 0.06246403232216835, -0.009584544226527214, 0.024432910606265068, 0.05734642595052719, -0.041671931743621826, 0.026494575664401054, -0.0...",[0.1653787446547943]
1,200017,B009NEQAHQ,A1CN3VUX1DQ0FN,5,"Great taste, beautiful crystal!",My wife and I use this everyday in our dinners. The crystal is a clean white finishing salt and with 33%lower sodium then table salt and 2% of our magnesium and calcium it is super healthy. I've used it in my water during workouts (just a little pinch) for a good electrolyte boost. i'm a big ...,"Title: Great taste, beautiful crystal!; Content: My wife and I use this everyday in our dinners. The crystal is a clean white finishing salt and with 33%lower sodium then table salt and 2% of our magnesium and calcium it is super healthy. I've used it in my water during workouts (just a little...",108,"[0.013699323870241642, 0.026146559044718742, -0.03358544036746025, 0.028989536687731743, 0.013124835677444935, -0.03924193233251572, -0.00035168963950127363, 0.011276164092123508, 0.003038156544789672, -0.03137587010860443, 0.04869888722896576, -0.04766775295138359, -0.04513411596417427, -0.0003...","[0.013699323870241642, 0.026146559044718742, -0.03358544036746025, 0.028989536687731743, 0.013124835677444935, -0.03924193233251572, -0.00035168963950127363, 0.011276164092123508, 0.003038156544789672, -0.03137587010860443, 0.04869888722896576, -0.04766775295138359, -0.04513411596417427, -0.0003...",[0.18961330005541593]
2,200016,B009NEQAHQ,AM2KJSYDRR0KI,4,Yummy!,"I love the taste of this salt. It works in all my recipes and is wonderful on raw veggies like cucumbers and tomatoes. I was very happy to learn that it was also healthy for me. It is in large chunks however, so you have to grind it if you are using it in baking. I used a mortar and pestle an...","Title: Yummy!; Content: I love the taste of this salt. It works in all my recipes and is wonderful on raw veggies like cucumbers and tomatoes. I was very happy to learn that it was also healthy for me. It is in large chunks however, so you have to grind it if you are using it in baking. I use...",107,"[0.030737943947315216, -0.00048475226503796875, -0.009015318937599659, 0.07166463136672974, 0.016613945364952087, 0.0038350881077349186, 0.011698449961841106, 0.0225526075810194, -0.01629912480711937, -0.010424857027828693, 0.053061593323946, -0.06284965574741364, 0.002330746268853545, 0.0201055...","[0.030737943947315216, -0.00048475226503796875, -0.009015318937599659, 0.07166463136672974, 0.016613945364952087, 0.0038350881077349186, 0.011698449961841106, 0.0225526075810194, -0.01629912480711937, -0.010424857027828693, 0.053061593323946, -0.06284965574741364, 0.002330746268853545, 0.0201055...",[0.12496596931671486]
3,112175,B0045H0KYA,A3RCEQLJPQXSB8,5,Good,Ni

In [20]:
df_with_sim_scores_sorted = df_with_sim_scores.sort_values("similarity", ascending=False)

In [21]:
df_with_sim_scores_sorted.columns

Index(['Unnamed: 0', 'ProductId', 'UserId', 'Score', 'Summary', 'Text',
       'combined', 'n_tokens', 'ada_embedding', 'embedding', 'similarity'],
      dtype='object')

In [22]:
pd.set_option('max_colwidth', 300)

In [23]:
# Selecting specific columns (e.g., 'column1', 'column2') from the sorted DataFrame
df_with_sim_scores_sorted[['combined', 'similarity']].sample(3)

,combined,similarity
1958,Title: BEST cup of coffee I've ever had!; Content: I thought I'd splurge and try this coffee. It costs much more than other decaf K-Cup options. But I hoped that meant it was better coffee. It IS better coffee. I've never had a better cup of coffee than this. It is excellent when compared t...,[0.28803136202014856]
1591,Title: Jamaican Blue beans; Content: Excellent coffee bean for roasting. Our family just purchased another 5 pounds for more roasting. Plenty of flavor and mild on acidity when roasted to a dark brown bean and before any oil appears on the bean itself (455F @ 17 minutes).,[0.5063571958820435]
455,"Title: This Is Fantastic; Content: Just got the Grove Square Caramel Apple Cider K-Cup order yesterday. Tasted it, loved it and promptly ordered an additional 2 boxes. I'd give this 10 stars if I could. The apple taste is pure, the caramel is spot on and the ease of use is outstanding. Thanks...",[0.2646253204837633]


In [24]:
# search through the reviews for a specific product
def search_reviews(df, query, n=5):
    
    response = client.embeddings.create(
                    model          = EMBEDDING_MODEL, 
                    input          = query, 
                    encoding_format= "float"
    )
    
    # obtain the embedding for the query
    query_embedding = np.array(response.data[0].embedding)[np.newaxis, :]
       
    df["similarity"] = df.embedding.apply(lambda x: get_cosine_similarity(x[np.newaxis, :], query_embedding))
    
    df_with_sim_scores_sorted = df_with_sim_scores.sort_values("similarity", ascending=False)
    
    # Selecting specific columns (e.g., 'column1', 'column2') from the sorted DataFrame
    results_df = df_with_sim_scores_sorted[['combined', 'similarity']].head(n)
    
    return results_df

In [25]:
search_reviews(df, "delicious beans", n=3)

,combined,similarity
266,"Title: Rancho Gordo Beans - What fun!; Content: I've probably tried about 15 varieties of beans from Rancho Gordo. They're all excellent. Some are better suited for one type dish than another, and if you visit their website, they will tell you what works best for which dish.<br /><br />Some ar...",[0.6128910652825382]
1340,"Title: Delicious!; Content: I enjoy this white beans seasoning, it gives a rich flavor to the beans I just love it, my mother in law didn't know about this Zatarain's brand and now she is traying different seasoning and she likes it very much.<br />Thank you Amazon for having it because now I ca...",[0.5735531186264144]
1495,Title: Fantastic Instant Refried beans; Content: Fantastic Instant Refried Beans have been a staple for my family now for nearly 20 years. All 7 of us love it and my grown kids are passing on the tradition.,[0.5581275989046359]


In [26]:
search_reviews(df, "whole wheat pasta", n=3)

,combined,similarity
879,"Title: Tasty and Quick Pasta; Content: Barilla Whole Grain Fusilli with Vegetable Marinara is tasty and has an excellent chunky vegetable marinara. I just wish there was more of it. If you aren't starving or on a diet, the 9oz serving is enough for lunch although you might want to add a piece ...",[0.49162131571936174]
1594,Title: sooo good; Content: tastes so good. Worth the money. My boyfriend hates wheat pasta and LOVES this. cooks fast tastes great.I love this brand and started buying more of their pastas. Bulk is best.,[0.489509216124933]
109,"Title: not gnocchi; Content: The package says that this pasta is gnocchi, but it's actually just whole wheat pasta in the shape of shells.",[0.47134547841412383]


In [27]:
search_reviews(df, "bad delivery", n=3)

,combined,similarity
1944,"Title: great product, poor delivery; Content: The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backe...",[0.4972110239196683]
1601,"Title: great product, poor delivery; Content: The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backe...",[0.4972110239196683]
1888,"Title: great product, poor delivery; Content: The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backe...",[0.4972110239196683]


In [28]:
search_reviews(df, "spoilt", n=1)

,combined,similarity
1555,Title: Disappointed; Content: The metal cover has severely disformed. And most of the cookies inside have been crushed into small pieces. Shopping experience is awful. I'll never buy it online again.,[0.3507283342740066]


In [29]:
search_reviews(df, "Rodeo Drive", n=2)

,combined,similarity
1869,Title: Rodeo Drive is Crazy Good Coffee!; Content: Rodeo Drive is my absolute favorite and I'm ready to order more! That's if I can find it.<br />I don't know why they are discontinuing it.<br />It arrived very fast.,[0.5124597291878411]
1471,Title: Rodeo Drive is Crazy Good Coffee!; Content: Rodeo Drive is my absolute favorite and I'm ready to order more! That's if I can find it.<br />I don't know why they are discontinuing it.<br />It arrived very fast.,[0.5124597291878411]


In [30]:
search_reviews(df, "Will switching to decaf tea at night helpful?", n=3)

,combined,similarity
1735,Title: breakfast tea; Content: We switch to this decaf tea at night for a great cup of tea and no sleep problems. Thanks for a good cup of tea.,[0.6277595024225457]
1880,Title: breakfast tea; Content: We switch to this decaf tea at night for a great cup of tea and no sleep problems. Thanks for a good cup of tea.,[0.6277097166689393]
1720,Title: breakfast tea; Content: We switch to this decaf tea at night for a great cup of tea and no sleep problems. Thanks for a good cup of tea.,[0.6277097166689393]


| **Method**                          | **Description**                                                                                           | **Advantages**                                | **Use Cases**                                                    |
|-------------------------------------|-----------------------------------------------------------------------------------------------------------|------------------------------------------------|------------------------------------------------------------------|
| **Cosine Similarity**                | Measures the cosine of the angle between two vectors.                                                     | Simple, effective, and widely used.            | General-purpose similarity search in embeddings.                 |
| **Dot Product (Inner Product)**      | Measures the direct product of two vectors.                                                               | Computationally efficient.                     | Tasks where embeddings are normalized.                           |
| **Euclidean Distance**               | Measures the straight-line distance between two vectors in space.                                         | Considers magnitude, simple to compute.        | Use cases where magnitude and scale matter.                      |
| **BM25 (Okapi BM25)**                | A ranking function based on term frequency and document length normalization.                             | Effective in traditional text retrieval.       | Initial retrieval in hybrid search systems.                      |
| **Approximate Nearest Neighbor (ANN)** | Techniques like LSH and HNSW for finding approximate nearest neighbors in large datasets.                 | Scales well to large datasets, fast retrieval. | Large-scale retrieval with trade-offs in accuracy.               |
| **Dual Encoder Models**             | Uses separate encoders for queries and documents and calculates similarity via learned functions.         | Effective for embedding-based retrieval.       | Initial retrieval or ranking tasks where embedding similarity is key. |
| **Cross-Encoder Models**             | Uses both query and document together to output a relevance score directly.                               | High accuracy in relevance scoring.            | Re-ranking top results for improved relevance.                   |
| **Attention Mechanisms**             | Dynamically weighs parts of the query and document embeddings to refine similarity scores.               | Handles complex retrieval tasks effectively.   | Multi-step reasoning and complex retrieval processes.            |


#### STOP HERE

#### Implement BM25

In [34]:
!pip install rank-bm25

In [35]:
from rank_bm25 import BM25Okapi
import numpy as np

In [36]:
# Function to preprocess and tokenize text
def tokenize(text):
    # Tokenization logic here, e.g., using simple whitespace split
    return text.lower().split()

In [37]:
# Prepare the corpus and query
def search_reviews_bm25(df, query, n=5):
    # Tokenize the corpus
    tokenized_corpus = [tokenize(doc) for doc in df['combined']]
    
    # Initialize BM25
    bm25 = BM25Okapi(tokenized_corpus)
    
    # Tokenize the query
    tokenized_query = tokenize(query)
    
    # Get BM25 scores
    scores = bm25.get_scores(tokenized_query)
    
    # Add scores to DataFrame
    df['bm25_score'] = scores
    
    # Sort by BM25 score and select top-n
    df_sorted = df.sort_values('bm25_score', ascending=False)
    
    # Selecting specific columns (e.g., 'combined', 'bm25_score') from the sorted DataFrame
    results_df = df_sorted[['combined', 'bm25_score']].head(n)
    
    return results_df

In [38]:
# Example usage
query = "delicious beans"

top_results = search_reviews_bm25(df, query)
top_results

,combined,bm25_score
1660,"Title: Best beans your money can buy; Content: These are, hands down, the best jelly beans on the market. There isn't a gross one in the bunch and each of them has an intense, delicious flavor. Though I hesitate to pick a favorite, I have to say that I love green apple, a rare flavor in drugst...",7.577846
1834,"Title: Panama green beans; Content: These beans have produced some enjoyable cups of coffee with a medium roast (dark brown beans with no oil on their surfaces) at 455F for 17 minutes. We have experimented, using the same roasted beans with different water sources. We have learned that the beans...",7.452268
1117,"Title: Amazing; Content: Nothing makes me feel at home more than a pot of blue runner red beans on the stove!! These red beans are the best. I'm so glad I can buy them here on amazon, my first two years of living away from Homs I had to bring them back with me or have someone send some! Now I ha...",6.231477
1340,"Title: Delicious!; Content: I enjoy this white beans seasoning, it gives a rich flavor to the beans I just love it, my mother in law didn't know about this Zatarain's brand and now she is traying different seasoning and she likes it very much.<br />Thank you Amazon for having it because now I ca...",6.069269
612,Title: coffee beans; Content: Coffee beans did not seem fresh. No oil on them what so ever. I have tasted much better and fresher. Will not order again.,5.926505


#### Key Points:
- **Tokenization:** The `tokenize` function should preprocess the text by converting it to lowercase and splitting it into tokens. You might use more sophisticated tokenization depending on your requirements.
- **BM25 Initialization:** `BM25Okapi` is initialized with the tokenized corpus.
- **Scoring:** BM25 scores are computed for the query and added to the DataFrame.
- **Sorting:** The DataFrame is sorted based on BM25 scores to retrieve the top-n results.

#### Explanation:
- **Tokenization:** Converts text to a list of tokens. This is crucial for BM25 to work effectively.
- **BM25 Scores:** Calculated using the `get_scores` method, which returns a list of scores for each document in the corpus.
- **Sorting and Selection:** The DataFrame is sorted based on the BM25 scores, and the top results are selected.


#### Implement Cross-Encoder Models

In [ ]:
#pip install sentence-transformers

In [39]:
from sentence_transformers import CrossEncoder

In [ ]:
# Load the pre-trained Cross-Encoder model
model_name = 'cross-encoder/ms-marco-TinyBERT-L-6'
model = CrossEncoder(model_name)

In [ ]:
# Function to compute scores for query-document pairs
def score_pairs(query, documents):
    # Create input pairs
    pairs = [(query, doc) for doc in documents]
    
    # Compute scores
    scores = model.predict(pairs)
    
    return scores

In [ ]:
# Function to search through the DataFrame using Cross-Encoder
def search_reviews_cross_encoder(df, query, n=5):
    # Extract documents
    documents = df['combined'].tolist()
    
    # Compute scores for query-document pairs
    scores = score_pairs(query, documents)
    
    # Add scores to DataFrame
    df['cross_encoder_score'] = scores
    
    # Sort by Cross-Encoder score and select top-n
    df_sorted = df.sort_values('cross_encoder_score', ascending=False)
    
    # Selecting specific columns (e.g., 'combined', 'cross_encoder_score') from the sorted DataFrame
    results_df = df_sorted[['combined', 'cross_encoder_score']].head(n)
    
    return results_df

In [ ]:
%%time
# Example usage
# can take more than an hour
query = "delicious beans"

top_results = search_reviews_cross_encoder(df, query)
top_results

#### Key Points:

- **Loading the Model:** 
  - `CrossEncoder` from `sentence-transformers` is used to load a pre-trained Cross-Encoder model. 
  - The `model_name` can be adjusted to any other suitable pre-trained Cross-Encoder model.

- **Scoring Function:** 
  - The `score_pairs` function generates query-document pairs and computes their relevance scores using the Cross-Encoder model.

- **Integration:** 
  - The `search_reviews_cross_encoder` function integrates this scoring mechanism into your search pipeline, sorting documents based on relevance scores.

#### Explanation:

- **Model:** 
  - The `CrossEncoder` model is used to jointly process query-document pairs and calculate relevance scores.

- **Pairing:** 
  - The `score_pairs` function pairs each document with the query and uses the model to compute a relevance score for each pair.

- **Sorting and Selection:** 
  - Scores are added to the DataFrame, which is then sorted based on these scores to retrieve the top results.

This approach provides a high-accuracy method for document retrieval by leveraging Cross-Encoder models to assess the relevance of documents with respect to a query.
